# Paligemma

[paligemma](pics/paligemma.png)

---

## 一、Configuration

In [ ]:
import torch
import torch.nn as nn


class SiglipVisionConfig:
    def __init__(
        self,
        hidden_size=768,
        intermediate_size=3072,
        num_hidden_layers=12,
        num_q_heads=12,
        num_channels=3,  # RGB三通道
        image_size=224,  # resize后的图像尺寸
        patch_size=16,   # 切割得到一个patch的边长
        layer_norm_eps=1e-16,
        attention_dropout=0.0,
        num_image_tokens: int = None,
        **kwargs,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_q_heads = num_q_heads
        self.num_channels = num_channels
        self.image_size = image_size
        self.patch_size = patch_size
        self.layer_norm_eps = layer_norm_eps
        self.attention_dropout = attention_dropout
        self.num_image_tokens = num_image_tokens
        

class GemmaConfig:
    def __init__(
        self,
        vocab_size,
        hidden_size,
        intermediate_size,
        num_hidden_layers,
        num_q_heads,
        num_kv_heads,  # Grouped-Query Attention
        head_dim=256,
        max_position_embeddings=8192,
        rms_norm_eps=1e-6,
        rope_theta=10000.0,
        attention_bias=False,
        attention_dropout=0.0,
        pad_token_id=None,
        **kwargs,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_position_embeddings = max_position_embeddings
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.attention_bias = attention_bias
        self.attention_dropout = attention_dropout
        self.pad_token_id = pad_token_id
        
        
class PaliGemmaConfig:
    def __init__(
        self,
        vision_config=None,
        text_config=None,
        ignore_index=-100,
        image_token_index=256000,
        vocab_size=257152,
        projection_dim=2048,  # projection的输出维度
        hidden_size=2048,     # 语言模型的嵌入维度
        pad_token_id=None,
        **kwargs,
    ):
        super().__init__()
        self.ignore_index = ignore_index
        self.image_token_index = image_token_index
        self.vocab_size = vocab_size
        self.projection_dim = projection_dim
        self.hidden_size = hidden_size
        self.vision_config = vision_config
        self.is_encoder_decoder = False
        self.pad_token_id = pad_token_id

        self.vision_config = SiglipVisionConfig(**vision_config)
        self.text_config = GemmaConfig(**text_config, pad_token_id=pad_token_id)
        self.vocab_size = self.text_config.vocab_size

        self.text_config.num_image_tokens = (self.vision_config.image_size // self.vision_config.patch_size)**2
        self.vision_config.projection_dim = projection_dim

---

## 二、SigLIP Embedding

In [ ]:
class SiglipVisionEmbeddings(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.image_size = config.image_size
        self.patch_size = config.patch_size
        
        """
        kernel_size = stride = patch_size, 卷积步之间无重叠
        保证输出的特征图长宽正好均为 image_size//stride=num_patches_per_dim
        """
        self.patch_embedding = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=self.embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
            padding="valid",  # no padding
        )
        
        self.num_patches = (self.image_size // self.patch_size) ** 2  # 将正方形切割为patch的集合
        self.num_positions = self.num_patches
        
        self.position_embedding = nn.Embedding(self.num_positions, self.embed_dim)  # learnable, 训练时更新
        self.register_buffer(
            "position_idx",
            torch.arange(self.num_positions).unsqueeze(0),
            persistent=False,
        )
        
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        通过卷积操作计算输入pixels的嵌入向量, 获取patch_embeds:
        (batch, channels, height, width) -> (batch, embed_dim, num_patches_H, num_patches_W)
        再将patch_embeds展平并将num_patches维往前提:
        (batch, embed_dim, num_patches_H, num_patches_W) -> (batch, embed_dim, num_patches) -> (batch, num_patches, embed_dim)
        """
        patch_embeds = self.patch_embedding(pixel_values)
        embeddings = patch_embeds.flatten(2).transpose(1, 2)
        
        """
        position_embedding: (1, num_patches) -> (1, num_patches, embed_dim)
        将线性位置编码叠加到patch_embeds上, 最终得到: (batch, num_patches, embed_dim)
        """
        embeddings = embeddings + self.position_embedding(self.position_idx)
        return embeddings

---

## 三、SigLIP VisionTransformer

In [ ]:
import torch.nn.functional as F

class SiglipMLP(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        (batch, num_patches, embed_dim) -> (batch, num_patches, intermediate_size) -> (batch, num_patches, embed_dim)
        """
        x = self.fc1(x)
        x = F.gelu(x, approximate="tanh")
        x = self.fc2(x)
        return x


class SiglipAttention(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.embed_dim // self.num_heads
        self.scale = self.head_dim ** -0.5  # 1 / sqrt(head_dim)
        self.dropout = config.attention_dropout
        
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim)
    
    """
    区别于LLM中的causal attention, SiglipAttention建立了所有patches之间的联系
    """
    def forward(self, x: torch.Tensor):
        batch_size, num_patches, embed_dim = x.size()
        """
        将同一个patched input分别投影至query, key, value, 形状不改变
        (batch, num_patches, embed_dim) -> (batch, num_patches, head_dim)
        """
        query = self.q_proj(x)
        key   = self.k_proj(x)
        value = self.v_proj(x)
        
        """
        将Q, K, V分别拆分成多个head, 再前置多头位置使每个head都可以看到所有的patches(的部分embed)
        (batch, num_patches, embed_dim) -> (batch, num_heads, num_patches, head_dim)
        """
        query = query.view(batch_size, self.num_heads, num_patches, self.head_dim)
        key   = key.view(batch_size, self.num_heads, num_patches, self.head_dim)
        value = value.view(batch_size, self.num_heads, num_patches, self.head_dim)
        
        """
        将多头Q乘以转置的K, 并作用softmax计算注意力分数:
        (batch, num_heads, num_patches, head_dim) @ (batch, num_heads, head_dim, num_patches) -> (batch, num_heads, num_patches, num_patches)
        再乘以多头V, 得到多头重建patches:
        (batch, num_heads, num_patches, num_patches) @ (batch, num_heads, num_patches, head_dim) -> (batch, num_heads, num_patches, head_dim)
        """
        attn_score  = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        attn_score  = F.softmax(attn_score, dim=-1, dtype=torch.float32)
        attn_score  = F.dropout(attn_score, p=self.dropout, training=self.training)
        attn_output = torch.matmul(attn_score, value)
        
        """
        将patches拼接重建成原始形状:
        (batch, num_heads, num_patches, head_dim) -> (batch, num_patches, num_heads * head_dim)
        再通过out_proj映射到输出空间, 形状不变:
        (batch, num_patches, num_heads * head_dim) -> (batch, num_patches, embed_dim)
        """
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, num_patches, self.embed_dim)
        attn_output = self.out_proj(attn_output)
        
        return attn_output
    
    
class SiglipEncoderLayer(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.embed_dim = config.hidden_size
        self.self_attn = SiglipAttention(config)
        self.layer_norm1 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.mlp = SiglipMLP(config)
        self.layer_norm2 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Block1: (batch, num_patches, embed_dim) -> (batch, num_patches, embed_dim), 形状始终不变
        """
        residual = x
        x = self.layer_norm1(x)
        x = self.self_attn(x)
        x = x + residual
        
        """
        Block2: (batch, num_patches, embed_dim) -> (batch, num_patches, embed_dim), 同上
        """
        residual = x
        x = self.layer_norm2(x)
        x = self.mlp(x)  # 扩充模型参数 + 引入非线性性, 支持建模更加复杂的关系
        x = x + residual
        
        return x


class SiglipEncoder(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.layers = nn.ModuleList(
            [SiglipEncoderLayer(config) for _ in range(config.num_hidden_layers)]
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        经过多个EncoderLayer, 形状始终不变: (batch, num_patches, embed_dim)
        """
        for encoder_layer in self.layers:
            x = encoder_layer(x)
        return x
        
    
class SiglipVisionTransformer(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        embed_dim = config.hidden_size
        self.embeddings = SiglipVisionEmbeddings(config)
        self.encoder = SiglipEncoder(config)
        self.post_layernorm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)
        
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        将输入图像转化为patch_embedding
        (batch, channels, height, width) -> (batch, num_patches, embed_dim)
        """
        hidden_states = self.embeddings(pixel_values)
        
        last_hidden_states = self.encoder(hidden_states)
        last_hidden_states = self.post_layernorm(last_hidden_states)
        
        return last_hidden_states
    
    
class SiglipVisionModel(nn.Module):
    def __init__(self, config: SiglipVisionConfig):
        super().__init__()
        self.config = config
        self.vit = SiglipVisionTransformer(config)
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        输入图像像素值, 输出图像嵌入特征
        (batch, channels, height, width) -> (batch, num_patches, embed_dim)
        """
        vit_output = self.vit(pixel_values=pixel_values)
        return vit_output

---

## 四、Vision Projection

In [ ]:
class PaliGemmaProjector(nn.Module):
    def __init__(self, config: PaliGemmaConfig):
        super().__init__()
        self.ori_dim = config.vision_config.hidden_size
        self.proj_dim = config.projection_dim
        self.projection = nn.Linear(self.ori_dim, self.proj_dim, bias=False)  # 一个简单的线性层
    
    def forward(self, image_features: torch.Tensor) -> torch.Tensor:
        """
        将图像嵌入特征投影到 LLM 嵌入维度
        (batch, num_patches, hidden_size) -> (batch, num_patches, projection_dim)
        """
        hidden_states = self.projection(image_features)
        return hidden_states

---

## 五、Mulitmodal Processing

In [ ]:
from PIL import Image
import numpy as np
from typing import List

IMAGENET_STANDARD_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STANDARD_STD = [0.5, 0.5, 0.5]

class PaliGemmaProcessor:    
    def __init__(self, tokenizer, num_image_tokens: int, image_size: int):
        super().__init__()
        self.tokenizer = tokenizer
        self.num_image_tokens = num_image_tokens
        self.image_size = image_size
        self.IMAGE_TOKEN = "[image]"
        
        self.image_token_id = tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)
        self.tokenizer.add_bos_token = False
        self.tokenizer.add_eos_token = False

    def process_images(
        self,
        images: Image.Image,
        size: tuple,
        resample: Image.Resampling = None,
        rescale_factor: float = None,
        image_mean: List[float] = None,
        image_std: List[float] = None,
    ):
        mean, std = np.array(image_mean, dtype=np.float32), np.array(image_std, dtype=np.float32)
        images = [image.resize(size=(size[0], size[1]), resample=resample) for image in images]
        images = [np.array(image).astype(np.float32) for image in images]
        images = [image * rescale_factor for image in images]
        images = [(image - mean) / std for image in images]
        images = [image.transpose(2, 0, 1) for image in images]  # (height, width, channels) -> (channels, height, width)
        
        pixel_values = np.stack(images, axis=0)  # (batch, channels, height, width)
        return torch.tensor(pixel_values, dtype=torch.float32)
    
    def add_image_tokens_to_prompt(
        prefix: str,
        bos_token: str,
        image_token: str,
        num_image_tokens: int,
    ):
        """
        构建预设提示, 包含image placeholder
        PaliGemma的实现形式为: [image]...[image][bos][prompt]\n
        """
        return f"{image_token * num_image_tokens}{bos_token}{prefix}\n"
    
    def __call__(
        self,
        texts: List[str],
        images: List[Image.Image],
        padding: str = "longest",
        truncation: bool = True,
    ):
        assert len(texts) == 1 and len(images) == 1, "Only one text and one image are supported."
        
        """
        将PIL图像转化为pixel tensor, 注意前后两者的*通道位置*不一样
        """
        pixel_values = self.process_images(
            images=images,
            size=(self.image_size, self.image_size),
            resample=Image.Resampling.BICUBIC,
            rescale_factor=1/255.0,
            image_mean=IMAGENET_STANDARD_MEAN,
            image_std=IMAGENET_STANDARD_STD,
        )
        
        """
        构建预设提示, 将image token插入text token中
        """
        input_strings = [
            self.add_image_tokens_to_prompt(
                prefix=prompt,
                bos_token=self.tokenizer.bos_token,
                image_token=self.IMAGE_TOKEN,
                num_image_tokens=self.num_image_tokens,
            )
            for prompt in texts
        ]
        
        """
        将文本和图像的输入字符转化为token id, 并以tensor形式返回 input_ids 和 attention_mask
        """
        inputs = self.tokenizer(
            input_strings,
            padding=padding,
            truncation=truncation,
            return_tensors="pt",
        )
        
        return_data = {"pixel_values": pixel_values, **inputs}
        return return_data

---

## 六、LLM KV-Cache

In [ ]:
from typing import List, Tuple

class KVCache:
    def __init__(self):
        """
        分别缓存key和value
        形状: (batch, num_kv_heads, seq_len, head_dim)
        """
        self.key_cache: List[torch.Tensor] = []
        self.value_cache: List[torch.Tensor] = []
        
    def num_items(self) -> int:
        if len(self.key_cache) == 0:
            return 0
        else:
            return self.key_cache[0].shape[2]  # return seq_len
        
    def update(
        self,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        layer_idx: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        更新kv-cache, 将当前的key和value*续接*入缓存中
        1. 如果当前层的cache是空的, 直接加入k和v即可
        2. 否则, 将输入的k和v与缓存中对应层的key和value拼接
        最终返回历史+当前输入的所有key和value
        """
        if len(self.key_cache) <= layer_idx:
            self.key_cache.append(key_states)
            self.value_cache.append(value_states)
        else:  # (batch, num_kv_heads, seq_len++, head_dim)
            self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_states], dim=-2)
            self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_states], dim=-2)
        
        return self.key_cache[layer_idx], self.value_cache[layer_idx]

---

## 七、RMSNorm

In [ ]:
class GemmaRMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.zeros(dim))  #(dim,)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float()
        """
        a_i' = (a_i / RMS(a)) * g_i
        (batch, seq_len, dim) * (dim,) -> (batch, seq_len, dim)
        """
        nomalized_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        nomalized_x = nomalized_x * self.weight.float()
        return nomalized_x.type_as(x)

---

## 八、Rotary Positional Embedding

> 具体原理及推导见 llama2.ipynb

In [ ]:
class GemmaRotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, seq_len: int = 2048, base=10000):
        super().__init__()
        self.head_dim = head_dim
        self.seq_len = seq_len
        self.base = base

    def precompute_theta_pos_frequencies(self, head_dim: int, seq_len: int, device: str):
        assert head_dim % 2 == 0, "head_dim must be even for RotaryEmbedding"

        """
        theta.shape: (dim/2)
        """
        i_iter = torch.arange(0, head_dim, 2).float()
        theta = 1.0 / (10000 ** (i_iter / head_dim)).to(device)

        """
        m.shape(position): (seq_len)
        freqs: (seq_len) @ (head_dim/2) -> (seq_len, head_dim/2), 即每个位置m都和所有的theta组合相乘, 得到正(余)弦内的数值
        """
        m = torch.arange(seq_len, device=device).float()
        freqs = torch.outer(m, theta).float()

        """
        torch.polar: 构建一个复数张量, 其元素模长均为1, 其元素角度来自freqs
        freqs_complex.shape: (seq_len, head_dim/2); 一个角度为mθ, 则对应的元素为cos(mθ)+i*sin(mθ)
        """
        freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
        return freqs_complex
    
    @torch.no_grad()
    def forward(self, x: torch.Tensor, device: str) -> torch.Tensor:
        """
        x.shape: (batch, n_heads, seq_len, head_dim)
        freqs_complex.shape: (seq_len, head_dim/2)
        """
        freqs_complex = self.precompute_theta_pos_frequencies(head_dim=x.shape[-1], seq_len=self.seq_len, device=device)

        """
        先利用view_as_complex将x的最后一维扩成2个 (对应一组实部+虚部)
            (batch, seq_len, n_heads, head_dim) -> (batch, seq_len, n_heads, head_dim/2, 2)
        再将x转化为复数, 与freqs_complex形状相同
            (batch, seq_len, n_heads, head_dim/2, 2) -> (batch, seq_len, n_heads, head_dim/2)
        """
        x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))

        """
        先扩展出批次和多头的维度, 便于与x_complex逐元素相乘
            (seq_len, head_dim/2) -> (1, seq_len, 1, head_dim/2)
        再利用广播机制与x_complex逐元素相乘
            (1, seq_len, 1, head_dim/2) * (batch, seq_len, n_heads, head_dim/2) -> (batch, seq_len, n_heads, head_dim/2)
        """
        freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)
        x_rotated = x_complex * freqs_complex

        """
        先利用view_as_real添加末尾2维度, 还原回实数张量
            (batch, seq_len, n_heads, head_dim/2) -> (batch, seq_len, n_heads, head_dim/2, 2)
        再拉直, 还原成输入x的形状, 得到旋转位置编码后的x
            (batch, seq_len, n_heads, head_dim/2, 2) -> (batch, seq_len, n_heads, head_dim)
        """
        x_out = torch.view_as_real(x_rotated)
        x_out = x_out.reshape(*x.shape)
        
        return x_out.type_as(x).to(device)

---

## 九、Decoder Layer

In [ ]:
class GemmaMLP(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.gelu(self.gate_proj(x)) * self.up_proj(x))

In [ ]:
from typing import Optional, Tuple
import math

class GemmaAttention(nn.Module):
    def __init__(self, config: GemmaConfig, layer_idx: Optional[int] = None):
        super().__init__()
        self.config = config
        self.attention_dropout = config.attention_dropout
        self.hidden_size = config.hidden_size

        self.num_q_heads = config.num_q_heads
        self.num_kv_heads = config.num_kv_heads
        self.head_dim = config.head_dim
        self.num_groups = self.num_q_heads // self.num_kv_heads

        self.rope_theta = config.rope_theta
        self.max_position_embeddings = config.max_position_embeddings

        assert self.hidden_size % self.num_q_heads == 0, "hidden_size must be divisible by num_q_heads"

        self.q_proj = nn.Linear(self.hidden_size, self.num_q_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_q_heads * self.head_dim, self.hidden_size, bias=config.attention_bias)

        self.rotary_emb = GemmaRotaryEmbedding(
            head_dim=self.head_dim,
            seq_len=self.max_position_embeddings,
            base=config.rope_theta,
        )
        self.layer_idx = layer_idx

    def repeat_kv(self, hidden_states: torch.Tensor, num_groups: int) -> torch.Tensor:
        batch_size, num_kv_heads, seq_len, head_dim = hidden_states.size()
        if num_groups == 1:
            return hidden_states
        """
        先扩展出中间的一个维度, 将其扩展到num_groups个组:
            (batch, num_kv_heads, seq_len, head_dim) -> (batch, num_kv_heads, num_groups, seq_len, head_dim)
        再将num_kv_heads进行复制, 使之与num_q_heads匹配:
            (batch, num_kv_heads, num_groups, seq_len, head_dim) -> (batch, num_q_heads, seq_len, head_dim)
        """
        hidden_states = hidden_states[:, :, None, :, :].expand(batch_size, num_kv_heads, num_groups, seq_len, head_dim)
        return hidden_states.reshape(batch_size, num_kv_heads*num_groups, seq_len, head_dim)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
        **kwargs,
    ) -> torch.Tensor:
        batch_size, seq_len, hidden_dim = hidden_states.size()

        """
        q: (batch, seq_len_q, hidden_dim) -> (batch, seq_len_q, num_q_heads * head_dim)
        kv: (batch, seq_len_kv, hidden_dim) -> (batch, seq_len_kv, num_kv_heads * head_dim)
        """
        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        """
        先拆分成多头并将多头维度调至序列维度前, 保证每个头都可以看到完整的sequence
        q: (batch, seq_len_q, num_q_heads * head_dim) -> (batch, num_q_heads, seq_len_q, head_dim)
        kv: (batch, seq_len_kv, num_kv_heads * head_dim) -> (batch, num_kv_heads, seq_len_kv, head_dim)
        """
        query_states = query_states.view(batch_size, seq_len, self.num_q_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        """
        仅对query和key应用旋转位置编码, 前后形状不变
        q: (batch, num_q_heads, seq_len_q, head_dim) -> (batch, num_q_heads, seq_len_q, head_dim)
        kv: (batch, num_kv_heads, seq_len_kv, head_dim) -> (batch, num_kv_heads, seq_len_kv, head_dim)
        """
        query_states = self.rotary_emb(query_states, device=hidden_states.device)
        key_states = self.rotary_emb(key_states, device=hidden_states.device)

        """
        将当前输入的k和v沿序列维度seq接入cache中, 并返回完整的历史kv序列
        """
        if kv_cache is not None:
            key_states, value_states = kv_cache.update(key_states, value_states, self.layer_idx)

        """
        将num_kv_heads进行复制, 使之与num_q_heads匹配
        kv: (batch, num_kv_heads, seq_len_kv, head_dim) -> (batch, num_q_heads, seq_len_kv, head_dim)
        """
        key_states = self.repeat_kv(key_states, self.num_groups)
        value_states = self.repeat_kv(value_states, self.num_groups)
        
        """
        多头注意力公式: (Q * K^T) / sqrt(head_dim)
        attn_weights: (batch, num_q_heads, seq_len_q, seq_len_kv)
        """
        attention_weights = torch.matmul(query_states, key_states.transpose(-1, -2)) / math.sqrt(self.head_dim)

        assert attention_mask is not None
        attention_weights = attention_weights + attention_mask

        attention_weights = F.dropout(
            attention_weights.softmax(dim=-1, dtype=torch.float32),  # (batch, num_q_heads, seq_len_q, seq_len_kv)
            p=self.attention_dropout, 
            training=self.training
        )

        """
        先将多头注意力分数乘以V
            (batch, num_q_heads, seq_len_q, seq_len_kv) @ (batch, num_q_heads, seq_len_kv, head_dim) -> (batch, num_q_heads, seq_len_q, head_dim)
        再将多头注意力的多个头拼接起来
            (batch, num_q_heads, seq_len_q, head_dim) -> (batch, seq_len_q, num_q_heads * head_dim) = (batch, seq_len_q, hidden_dim)
        """
        attention_weights = torch.matmul(attention_weights, value_states)
        attention_weights = attention_weights.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)

        """
        最终乘以o_proj, 得到多头注意力输出
        (batch, seq_len, hidden_dim) -> (batch, seq_len, hidden_dim)
        """
        attention_output = self.o_proj(attention_weights)

        return attention_output

In [ ]:
class GemmaDecoderLayer(nn.Module):
    def __init__(self, config: GemmaConfig, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.self_attn = GemmaAttention(config=config, layer_idx=layer_idx)
        self.mlp = GemmaMLP(config=config)
        self.input_attn_layernorm = GemmaRMSNorm(dim=self.hidden_size, eps=config.rms_norm_eps)
        self.post_attn_layernorm = GemmaRMSNorm(dim=self.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
    ) -> torch.Tensor:
        residual = hidden_states
        # (batch, seq_len, hidden_dim)
        hidden_states = self.input_attn_layernorm(hidden_states)

        # (batch, seq_len, hidden_dim)
        hidden_states = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            kv_cache=kv_cache
        )
        hidden_states = hidden_states + residual

        # (batch, seq_len, hidden_dim)
        residual = hidden_states
        hidden_states = self.post_attn_layernorm(hidden_states)

        # (batch, seq_len, hidden_dim)
        hidden_states = self.mlp(hidden_states)
        hidden_states = hidden_states + residual

        return hidden_states

---

## 十、GemmaForCausalLM

In [ ]:
class GemmaModel(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.config = config
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)
        self.layers = nn.ModuleList(
            [GemmaDecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)]
        )
        self.norm = GemmaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        inputs_embeds: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
    ) -> torch.FloatTensor:
        # (batch, seq_len, hidden_dim)
        hidden_states = inputs_embeds
        normalizer = torch.tensor(self.config.hidden_size**5, dtype=hidden_states.dtype)
        hidden_states = hidden_states * normalizer

        # (batch, seq_len, hidden_dim)
        for decoder_layer in self.layers:
            hidden_states = decoder_layer(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                kv_cache=kv_cache,
            )

        # (batch, seq_len, hidden_dim)
        hidden_states = self.norm(hidden_states)
        return hidden_states

In [ ]:
class GemmaForCausalLM(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.model = GemmaModel(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

    def forward(
        self,
        inputs_embeds: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
    ) -> Tuple:
        # (batch, seq_len, hidden_dim)
        hidden_states = self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            kv_cache=kv_cache,
        )

        # (batch, seq_len, vocab_size)
        logits = self.lm_head(hidden_states).float()
        
        return_data = {
            "logits": logits
        }

        if kv_cache is not None:
            return_data["kv_cache"] = kv_cache

        return return_data